In [1]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split

In [7]:
load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise RuntimeError("MONGODB_URI not found.")

In [11]:
client = MongoClient(MONGODB_URI)

db = client["aqi_predictor"]

collection = db["aqi_features"]

df = pd.DataFrame(list(collection.find()))

df.head()

,_id,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,6a6f6293b5806e9b28be5995,Karachi,2026-07-03T16:00:00+00:00,16,3,7,4,65,0.0,18.76,82.07,44.41,0.08,0.41,70.95,29.7,78,1000.9,15.2
1,6a6f6294b5806e9b28be5996,Karachi,2026-07-03T17:00:00+00:00,17,3,7,4,65,0.0,18.60,80.65,44.21,0.08,0.42,70.65,29.5,79,1001.5,17.0
2,6a6f6294b5806e9b28be5997,Karachi,2026-07-03T18:00:00+00:00,18,3,7,4,65,0.0,18.60,80.48,43.39,0.09,0.44,70.06,29.4,78,1001.5,17.2
3,6a6f6294b5806e9b28be5998,Karachi,2026-07-03T19:00:00+00:00,19,3,7,4,64,-1.0,18.50,78.31,42.68,0.09,0.45,69.26,29.2,80,1001.5,15.5
4,6a6f6294b5806e9b28be5999,Karachi,2026-07-03T20:00:00+00:00,20,3,7,4,65,1.0,18.55,76.07,42.22,0.09,0.46,68.63,29.0,81,1001.1,15.7


In [12]:
if "_id" in df.columns:
    df.drop(columns="_id", inplace=True)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,Karachi,2026-07-03T16:00:00+00:00,16,3,7,4,65,0.0,18.76,82.07,44.41,0.08,0.41,70.95,29.7,78,1000.9,15.2
1,Karachi,2026-07-03T17:00:00+00:00,17,3,7,4,65,0.0,18.60,80.65,44.21,0.08,0.42,70.65,29.5,79,1001.5,17.0
2,Karachi,2026-07-03T18:00:00+00:00,18,3,7,4,65,0.0,18.60,80.48,43.39,0.09,0.44,70.06,29.4,78,1001.5,17.2
3,Karachi,2026-07-03T19:00:00+00:00,19,3,7,4,64,-1.0,18.50,78.31,42.68,0.09,0.45,69.26,29.2,80,1001.5,15.5
4,Karachi,2026-07-03T20:00:00+00:00,20,3,7,4,65,1.0,18.55,76.07,42.22,0.09,0.46,68.63,29.0,81,1001.1,15.7


In [13]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")

df.reset_index(drop=True, inplace=True)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,Karachi,2026-07-03 16:00:00+00:00,16,3,7,4,65,0.0,18.76,82.07,44.41,0.08,0.41,70.95,29.7,78,1000.9,15.2
1,Karachi,2026-07-03 17:00:00+00:00,17,3,7,4,65,0.0,18.60,80.65,44.21,0.08,0.42,70.65,29.5,79,1001.5,17.0
2,Karachi,2026-07-03 18:00:00+00:00,18,3,7,4,65,0.0,18.60,80.48,43.39,0.09,0.44,70.06,29.4,78,1001.5,17.2
3,Karachi,2026-07-03 19:00:00+00:00,19,3,7,4,64,-1.0,18.50,78.31,42.68,0.09,0.45,69.26,29.2,80,1001.5,15.5
4,Karachi,2026-07-03 20:00:00+00:00,20,3,7,4,65,1.0,18.55,76.07,42.22,0.09,0.46,68.63,29.0,81,1001.1,15.7


In [14]:
FORECAST_HOURS = 72

df["target_aqi"] = df["aqi"].shift(-FORECAST_HOURS)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,target_aqi
0,Karachi,2026-07-03 16:00:00+00:00,16,3,7,4,65,0.0,18.76,82.07,44.41,0.08,0.41,70.95,29.7,78,1000.9,15.2,88.0
1,Karachi,2026-07-03 17:00:00+00:00,17,3,7,4,65,0.0,18.60,80.65,44.21,0.08,0.42,70.65,29.5,79,1001.5,17.0,89.0
2,Karachi,2026-07-03 18:00:00+00:00,18,3,7,4,65,0.0,18.60,80.48,43.39,0.09,0.44,70.06,29.4,78,1001.5,17.2,90.0
3,Karachi,2026-07-03 19:00:00+00:00,19,3,7,4,64,-1.0,18.50,78.31,42.68,0.09,0.45,69.26,29.2,80,1001.5,15.5,92.0
4,Karachi,2026-07-03 20:00:00+00:00,20,3,7,4,65,1.0,18.55,76.07,42.22,0.09,0.46,68.63,29.0,81,1001.1,15.7,93.0


In [15]:
df = df.dropna(subset=["target_aqi"])

df.reset_index(drop=True, inplace=True)

df.tail()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,target_aqi
619,Karachi,2026-07-30 11:00:00+00:00,11,30,7,3,87,-1.0,29.23,123.23,43.37,0.07,0.58,86.75,31.8,70,995.2,12.2,75.0
620,Karachi,2026-07-30 12:00:00+00:00,12,30,7,3,86,-1.0,28.70,120.75,43.67,0.07,0.59,84.79,31.4,72,995.1,13.5,75.0
621,Karachi,2026-07-30 13:00:00+00:00,13,30,7,3,85,-1.0,28.34,119.89,44.19,0.09,0.60,83.43,30.4,77,995.1,13.2,74.0
622,Karachi,2026-07-30 14:00:00+00:00,14,30,7,3,85,0.0,28.07,119.07,44.64,0.11,0.61,82.61,29.7,81,995.5,10.7,74.0
623,Karachi,2026-07-30 15:00:00+00:00,15,30,7,3,84,-1.0,27.97,119.28,45.20,0.12,0.62,82.55,29.2,84,996.0,9.6,74.0


In [16]:
print(df.shape)

df.info()

(624, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 624 entries, 0 to 623
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   city             624 non-null    object             
 1   timestamp        624 non-null    datetime64[ns, UTC]
 2   hour             624 non-null    int64              
 3   day              624 non-null    int64              
 4   month            624 non-null    int64              
 5   day_of_week      624 non-null    int64              
 6   aqi              624 non-null    int64              
 7   aqi_change_rate  624 non-null    float64            
 8   pm25             624 non-null    float64            
 9   pm10             624 non-null    float64            
 10  o3               624 non-null    float64            
 11  no2              624 non-null    float64            
 12  so2              624 non-null    float64            
 13  co        

In [17]:
FEATURES = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "pm25",
    "pm10",
    "o3",
    "no2",
    "so2",
    "co",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
]

In [18]:
X = df[FEATURES]

y = df["target_aqi"]

print(X.shape)
print(y.shape)

(624, 14)
(624,)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
)

In [20]:
print("Training")

print(X_train.shape)
print(y_train.shape)

print()

print("Testing")

print(X_test.shape)
print(y_test.shape)

Training
(499, 14)
(499,)

Testing
(125, 14)
(125,)


In [21]:
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import numpy as np

In [22]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [23]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[79.19438006 81.32132936 81.80919948 79.47763542 76.24866963 75.7371896
 72.78806003 81.38242583 80.54776837 83.91477064]


In [24]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 20.72
RMSE : 25.42
R²   : -18.3961


In [25]:
results = X_test.copy()

results["Actual AQI"] = y_test.values

results["Predicted AQI"] = y_pred

results.head(20)

,hour,day,month,day_of_week,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,Actual AQI,Predicted AQI
499,11,25,7,5,21.78,105.21,41.10,0.11,0.47,66.04,31.0,69,999.1,14.4,78.0,79.194380
500,12,25,7,5,21.81,104.74,40.83,0.11,0.48,65.97,30.5,71,998.7,11.9,78.0,81.321329
501,13,25,7,5,21.89,104.30,40.45,0.12,0.50,66.00,29.6,75,999.0,10.5,78.0,81.809199
502,14,25,7,5,22.02,103.72,39.74,0.13,0.51,66.00,29.2,78,999.7,10.8,78.0,79.477635
503,15,25,7,5,22.11,102.97,39.09,0.13,0.52,66.14,28.9,80,999.8,12.3,78.0,76.248670
504,16,25,7,5,22.10,101.77,38.52,0.13,0.52,66.31,28.9,80,1000.7,12.0,78.0,75.737190
505,17,25,7,5,22.12,101.09,38.18,0.13,0.52,66.46,28.9,80,1001.0,14.5,78.0,72.788060
506,18,25,7,5,22.43,102.09,37.83,0.12,0.53,66.64,28.4,79,1001.2,9.2,78.0,81.382426
507,19,25,7,5,22.81,103.92,37.41,0.12,0.55,66.68,28.2,80,1000.6,9.0,78.0,80.547768
508,20,25,7,5,22.86,104.36,37.16,0.12,0.55,66.84,28.2,79,1000.3,6.4,79.0,83.914771


In [26]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_
})

importance = importance.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False,
)

importance

,Feature,Coefficient
8,so2,-9.790037e+01
7,no2,-2.825264e+01
10,temperature,-6.065943e+00
13,wind_speed,-9.507252e-01
3,day_of_week,9.382954e-01
11,humidity,-9.233575e-01
9,co,-7.787385e-01
1,day,-6.967282e-01
5,pm10,6.408314e-01
4,pm25,-5.270264e-01


In [27]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/linear_regression.pkl")

print("Model saved successfully.")

Model saved successfully.
